In [ ]:
#Installing necessary libraries
!pip install faiss-cpu pandas
!pip install datasets -q

In [ ]:
#Mount google drive, if needed
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import pickle
import re
import json

PATH_MEGA_RAG_PKL = 'MEGA-RAG_pkl_file_path'  #Path for PKL file
MILU_TRANSLATION_PATHS = {
    'Kannada (Val)': 'stage_1_translated.json_path',
    'Marathi (Val)': 'stage_1_translated.json_path',
    'Telugu (Val)': 'stage_1_translated.json_path',
    'Malayalam (Test)': 'Translated_csv_file_path',
    'Odia (Test)': 'Translated_csv_file_path',
    'Punjabi (Test)': 'Translated_csv_file_path',
    'Tamil (Test)': 'Translated_csv_file_path',
    'Hindi (Test)': 'Translated_csv_file_path'
}

N_GRAM_SIZE = 8
TARGET_COL = 'question_en'  #Column name where translated questions stored

def clean_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    return ' '.join(text.split())

def get_ngrams(text, n=N_GRAM_SIZE):
    words = text.split()
    if len(words) < n:
        return set([text])
    return set([' '.join(words[i:i+n]) for i in range(len(words)-n+1)])

def extract_texts_from_pkl(pkl_data):
    if isinstance(pkl_data, pd.DataFrame):
        if 'question' in pkl_data.columns: return pkl_data['question'].tolist()
        if 'text' in pkl_data.columns: return pkl_data['text'].tolist()
        return pkl_data.iloc[:, 0].tolist()
    elif isinstance(pkl_data, list):
        if len(pkl_data) > 0 and isinstance(pkl_data[0], dict):
            key = 'question' if 'question' in pkl_data[0] else 'text'
            return [str(item.get(key, "")) for item in pkl_data]
        elif len(pkl_data) > 0 and isinstance(pkl_data[0], str):
            return pkl_data
    elif isinstance(pkl_data, dict):
        first_val = next(iter(pkl_data.values()))
        if isinstance(first_val, str):
            return list(pkl_data.values())
        elif isinstance(first_val, dict):
            key = 'question' if 'question' in first_val else 'text'
            return [str(val.get(key, "")) for val in pkl_data.values()]
    return []

def run_translation_audit():
    print("Loading MEGA-RAG PKL Database...")
    try:
        with open(PATH_MEGA_RAG_PKL, 'rb') as f:
            rag_data = pickle.load(f)
    except Exception as e:
        print(f"Error loading PKL file: {e}")
        return

    rag_questions_raw = extract_texts_from_pkl(rag_data)
    rag_questions_clean = [clean_text(q) for q in rag_questions_raw if clean_text(q)]
    total_rag = len(rag_questions_clean)
    print(f"Loaded {total_rag} cleaned documents from MEGA-RAG.\n")

    results_report = []
    overall_milu_questions = set()
    overall_milu_ngrams = set()

    for lang_split, path in MILU_TRANSLATION_PATHS.items():
        print(f"Processing {lang_split}...")
        try:
            if path.endswith('.json'):
                df = pd.read_json(path)
            elif path.endswith('.csv'):
                df = pd.read_csv(path)
            elif path.endswith('.xlsx'):
                df = pd.read_excel(path)
            else:
                print(f"  -> Unsupported file type for {path}. Skipping.")
                continue
        except Exception as e:
            print(f"  -> Error loading {path}: {e}")
            continue

        if TARGET_COL not in df.columns:
            print(f"  -> Error: Column/Key '{TARGET_COL}' not found in {lang_split}. Skipping.")
            continue

        milu_q_clean = [clean_text(q) for q in df[TARGET_COL].tolist() if clean_text(q)]

        overall_milu_questions.update(milu_q_clean)

        lang_ngrams = set()
        for q in milu_q_clean:
            ngrams = get_ngrams(q)
            lang_ngrams.update(ngrams)
            overall_milu_ngrams.update(ngrams)

        lang_exact = 0
        lang_contam = 0

        for rag_q in rag_questions_clean:
            if rag_q in milu_q_clean:
                lang_exact += 1
                lang_contam += 1
                continue

            rag_ngrams = get_ngrams(rag_q)
            overlap = rag_ngrams.intersection(lang_ngrams)
            if len(overlap) > 0 and len(rag_ngrams) > 0:
                if len(overlap) / len(rag_ngrams) >= 0.80:
                    lang_contam += 1

        lang_rate = (lang_contam / total_rag) * 100
        results_report.append({
            'Language/Split': lang_split,
            'Exact Matches': lang_exact,
            'Contaminated': f"{lang_contam} / {total_rag}",
            'Rate (%)': round(lang_rate, 5)
        })

    print("\nCalculating True Overall Contamination (Global Sweep)...")
    overall_exact = 0
    overall_contaminated = 0

    for rag_q in rag_questions_clean:
        if rag_q in overall_milu_questions:
            overall_exact += 1
            overall_contaminated += 1
            continue

        rag_ngrams = get_ngrams(rag_q)
        overlap = rag_ngrams.intersection(overall_milu_ngrams)
        if len(overlap) > 0 and len(rag_ngrams) > 0:
            if len(overlap) / len(rag_ngrams) >= 0.80:
                overall_contaminated += 1

    overall_rate = (overall_contaminated / total_rag) * 100

    print("\n=================================================================")
    print("FINAL TRANSLATION CONTAMINATION REPORT")
    print("=================================================================")
    if results_report:
        report_df = pd.DataFrame(results_report)
        print(report_df.to_string(index=False))
    else:
        print("No valid datasets processed.")
    print("-----------------------------------------------------------------")
    print(f"OVERALL EXACT MATCHES: {overall_exact}")
    print(f"OVERALL CONTAMINATED (Jaccard >= 0.80): {overall_contaminated} / {total_rag}")
    print(f"OVERALL CONTAMINATION RATE: {overall_rate:.6f}%")
    print("=================================================================")

run_translation_audit()

OUTPUT

In [ ]:
#Copy of Output cell of above code for our database
Loading MEGA-RAG PKL Database...
Loaded 546870 cleaned documents from MEGA-RAG.

Processing Kannada (Val)...
Processing Marathi (Val)...
Processing Telugu (Val)...
Processing Malayalam (Test)...
Processing Odia (Test)...
Processing Punjabi (Test)...
Processing Tamil (Test)...
Processing Hindi (Test)...

Calculating True Overall Contamination (Global Sweep)...

=================================================================
FINAL TRANSLATION CONTAMINATION REPORT
=================================================================
  Language/Split  Exact Matches Contaminated  Rate (%)
   Kannada (Val)             22  28 / 546870   0.00512
   Marathi (Val)             22  28 / 546870   0.00512
    Telugu (Val)             20  20 / 546870   0.00366
Malayalam (Test)             24  30 / 546870   0.00549
     Odia (Test)             28  35 / 546870   0.00640
  Punjabi (Test)             28  29 / 546870   0.00530
    Tamil (Test)             22  34 / 546870   0.00622
    Hindi (Test)             63  67 / 546870   0.01225
-----------------------------------------------------------------
OVERALL EXACT MATCHES: 124
OVERALL CONTAMINATED (Jaccard >= 0.80): 135 / 546870
OVERALL CONTAMINATION RATE: 0.024686%
=================================================================